In [1]:
!pip install -q sentence-transformers faiss-cpu transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 55.0 MB/s eta 0:00:00


In [2]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [3]:
documents = [
    """
    Retrieval Augmented Generation (RAG) is a technique that combines
    information retrieval with text generation. It retrieves relevant
    information from a knowledge source and provides it to a language model
    as context before generating an answer.
    """,

    """
    RAG can reduce hallucination because the language model receives
    factual information from an external knowledge source before answering.
    However, RAG does not completely eliminate hallucinations.
    """,

    """
    The main steps in a RAG pipeline are document loading, text chunking,
    embedding generation, vector storage, similarity search, and answer
    generation using the retrieved context.
    """,

    """
    Prompt engineering is the process of designing effective instructions
    for a language model. A good prompt can specify the task, provide context,
    define constraints, and instruct the model to avoid unsupported claims.
    """,

    """
    Embeddings represent text as numerical vectors. Similar pieces of text
    have vectors that are close to each other in the embedding space.
    These vectors can be stored in a vector database for efficient retrieval.
    """
]

print("Number of documents:", len(documents))

Number of documents: 5


In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(documents)

print("Embedding shape:", embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding shape: (5, 384)


In [5]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings).astype("float32"))

print("Documents stored in FAISS:", index.ntotal)

Documents stored in FAISS: 5


In [6]:
def retrieve_documents(query, k=2):
    query_embedding = model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding).astype("float32"),
        k
    )

    results = []

    for i in indices[0]:
        results.append(documents[i])

    return results

In [7]:
query = "How does RAG reduce hallucination?"

results = retrieve_documents(query)

for i, result in enumerate(results):
    print(f"\nResult {i+1}:")
    print(result)


Result 1:

    RAG can reduce hallucination because the language model receives
    factual information from an external knowledge source before answering.
    However, RAG does not completely eliminate hallucinations.
    

Result 2:

    Retrieval Augmented Generation (RAG) is a technique that combines
    information retrieval with text generation. It retrieves relevant
    information from a knowledge source and provides it to a language model
    as context before generating an answer.
    


In [8]:
def create_prompt(query, context):

    context_text = "\n\n".join(context)

    prompt = f"""
You are a helpful question-answering assistant.

Answer the question using ONLY the information provided in the context.

Rules:
1. Do not use outside knowledge.
2. Do not make up facts.
3. If the answer is not present in the context, say:
   "I don't have enough information to answer this question."
4. Give a concise and factual answer.

Context:
{context_text}

Question:
{query}

Answer:
"""

    return prompt

In [9]:
query = "How does RAG reduce hallucination?"

context = retrieve_documents(query)

prompt = create_prompt(query, context)

print(prompt)


You are a helpful question-answering assistant.

Answer the question using ONLY the information provided in the context.

Rules:
1. Do not use outside knowledge.
2. Do not make up facts.
3. If the answer is not present in the context, say:
   "I don't have enough information to answer this question."
4. Give a concise and factual answer.

Context:

    RAG can reduce hallucination because the language model receives
    factual information from an external knowledge source before answering.
    However, RAG does not completely eliminate hallucinations.
    


    Retrieval Augmented Generation (RAG) is a technique that combines
    information retrieval with text generation. It retrieves relevant
    information from a knowledge source and provides it to a language model
    as context before generating an answer.
    

Question:
How does RAG reduce hallucination?

Answer:



In [10]:
!pip install -q transformers torch

In [11]:
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="distilgpt2"
)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [12]:
output = generator(
    prompt,
    max_new_tokens=100,
    do_sample=False
)

print(output[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



You are a helpful question-answering assistant.

Answer the question using ONLY the information provided in the context.

Rules:
1. Do not use outside knowledge.
2. Do not make up facts.
3. If the answer is not present in the context, say:
   "I don't have enough information to answer this question."
4. Give a concise and factual answer.

Context:

    RAG can reduce hallucination because the language model receives
    factual information from an external knowledge source before answering.
    However, RAG does not completely eliminate hallucinations.
    


    Retrieval Augmented Generation (RAG) is a technique that combines
    information retrieval with text generation. It retrieves relevant
    information from a knowledge source and provides it to a language model
    as context before generating an answer.
    

Question:
How does RAG reduce hallucination?

Answer:
1. Do not use outside knowledge.
2. Do not make up facts.
3. If the answer is not present in the context, say:   

In [13]:
query = "Who invented RAG?"

context = retrieve_documents(query)

prompt = create_prompt(query, context)

print(prompt)


You are a helpful question-answering assistant.

Answer the question using ONLY the information provided in the context.

Rules:
1. Do not use outside knowledge.
2. Do not make up facts.
3. If the answer is not present in the context, say:
   "I don't have enough information to answer this question."
4. Give a concise and factual answer.

Context:

    Retrieval Augmented Generation (RAG) is a technique that combines
    information retrieval with text generation. It retrieves relevant
    information from a knowledge source and provides it to a language model
    as context before generating an answer.
    


    RAG can reduce hallucination because the language model receives
    factual information from an external knowledge source before answering.
    However, RAG does not completely eliminate hallucinations.
    

Question:
Who invented RAG?

Answer:



In [14]:
normal_prompt = """
Who invented RAG?
Answer the question confidently.
"""

output = generator(
    normal_prompt,
    max_new_tokens=50,
    do_sample=False
)

print(output[0]["generated_text"])

[transformers] Both `max_new_tokens` (=50) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Who invented RAG?
Answer the question confidently.
Rag is a term that is used to describe a person who is a person who is a person who is a person who is a person who is a person who is a person who is a person who is a person who is a person who is


In [15]:
query = "Who invented RAG?"

context = retrieve_documents(query)

rag_prompt = create_prompt(query, context)

output = generator(
    rag_prompt,
    max_new_tokens=100,
    do_sample=False
)

print(output[0]["generated_text"])

[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a helpful question-answering assistant.

Answer the question using ONLY the information provided in the context.

Rules:
1. Do not use outside knowledge.
2. Do not make up facts.
3. If the answer is not present in the context, say:
   "I don't have enough information to answer this question."
4. Give a concise and factual answer.

Context:

    Retrieval Augmented Generation (RAG) is a technique that combines
    information retrieval with text generation. It retrieves relevant
    information from a knowledge source and provides it to a language model
    as context before generating an answer.
    


    RAG can reduce hallucination because the language model receives
    factual information from an external knowledge source before answering.
    However, RAG does not completely eliminate hallucinations.
    

Question:
Who invented RAG?

Answer:
                                                                                                    


In [16]:
def rag_pipeline(query):

    # Retrieve relevant documents
    context = retrieve_documents(query)

    # Create grounded prompt
    prompt = create_prompt(query, context)

    # Generate answer
    output = generator(
        prompt,
        max_new_tokens=100,
        do_sample=False
    )

    return output[0]["generated_text"]

In [17]:
question = "What are the main steps in a RAG pipeline?"

answer = rag_pipeline(question)

print(answer)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



You are a helpful question-answering assistant.

Answer the question using ONLY the information provided in the context.

Rules:
1. Do not use outside knowledge.
2. Do not make up facts.
3. If the answer is not present in the context, say:
   "I don't have enough information to answer this question."
4. Give a concise and factual answer.

Context:

    The main steps in a RAG pipeline are document loading, text chunking,
    embedding generation, vector storage, similarity search, and answer
    generation using the retrieved context.
    


    Retrieval Augmented Generation (RAG) is a technique that combines
    information retrieval with text generation. It retrieves relevant
    information from a knowledge source and provides it to a language model
    as context before generating an answer.
    

Question:
What are the main steps in a RAG pipeline?

Answer:
1. Do not use outside knowledge.
2. Do not make up facts.
3. If the answer is not present in the context, say:
   "I don't 